In [1]:
!pip install pandas numpy pymatgen matminer scikit-learn

In [2]:
import pandas as pd
import numpy as np

# Path to your cleaned dataset
file_path = r"C:\Users\Atul\materials_project_cleaning\materials_cleaned.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset loaded successfully!
Shape: (10000, 5)

Columns:
['material_id', 'chemical_formula', 'crystal_system', 'space_group', 'full_structure']

First 5 rows:


,material_id,chemical_formula,crystal_system,space_group,full_structure
0,mp-3346392,CaMg(CO3)2,Monoclinic,P2_1/c,"{""@module"": ""pymatgen.core.structure"", ""@class..."
1,mp-3346394,Ca2Mg(CO3)3,Monoclinic,C2/c,"{""@module"": ""pymatgen.core.structure"", ""@class..."
2,mp-3346398,CaMg(CO3)2,Triclinic,P-1,"{""@module"": ""pymatgen.core.structure"", ""@class..."
3,mp-3346400,Li4Mg3Cl10,Monoclinic,Cm,"{""@module"": ""pymatgen.core.structure"", ""@class..."
4,mp-3346401,CaMg3(CO3)4,Triclinic,P-1,"{""@module"": ""pymatgen.core.structure"", ""@class..."


In [3]:
# ============================================================
# FIND CHEMICAL FORMULA COLUMN
# ============================================================

formula_candidates = [
    "chemical_formula",
    "formula",
    "formula_pretty",
    "Formula",
    "composition",
    "Composition"
]

formula_col = None

for col in formula_candidates:
    if col in df.columns:
        formula_col = col
        break


# ============================================================
# FIND CRYSTAL SYSTEM TARGET COLUMN
# ============================================================

target_candidates = [
    "crystal_system",
    "Crystal System",
    "crystal system",
    "symmetry"
]

target_col = None

for col in target_candidates:
    if col in df.columns:
        target_col = col
        break


print("Chemical formula column:", formula_col)
print("Crystal system column:", target_col)

Chemical formula column: chemical_formula
Crystal system column: crystal_system


In [4]:
print("Crystal System Classes:")
print(df[target_col].value_counts())

print("\nNumber of classes:",
      df[target_col].nunique())

Crystal System Classes:
crystal_system
Triclinic       4230
Monoclinic      2601
Trigonal        1027
Orthorhombic     911
Cubic            620
Tetragonal       540
Hexagonal         71
Name: count, dtype: int64

Number of classes: 7


In [5]:
from pymatgen.core import Composition

# ============================================================
# CONVERT FORMULA TO PYMATGEN COMPOSITION
# ============================================================

def convert_to_composition(formula):
    try:
        return Composition(formula)
    except:
        return None


df["composition"] = df[formula_col].apply(convert_to_composition)

print("Composition conversion completed.")

print("\nExample:")
display(df[[formula_col, "composition"]].head())

Composition conversion completed.

Example:


,chemical_formula,composition
0,CaMg(CO3)2,"(Ca, Mg, C, O)"
1,Ca2Mg(CO3)3,"(Ca, Mg, C, O)"
2,CaMg(CO3)2,"(Ca, Mg, C, O)"
3,Li4Mg3Cl10,"(Li, Mg, Cl)"
4,CaMg3(CO3)4,"(Ca, Mg, C, O)"


In [6]:
before = len(df)

df = df.dropna(
    subset=["composition", target_col]
).reset_index(drop=True)

after = len(df)

print("Rows before:", before)
print("Rows after:", after)
print("Invalid rows removed:", before - after)

Rows before: 10000
Rows after: 10000
Invalid rows removed: 0


In [7]:
from matminer.featurizers.composition import ElementProperty

# ============================================================
# MATMINER MAGPIE FEATURES
# ============================================================

featurizer = ElementProperty.from_preset("magpie")

print("Generating Magpie features...")

df_features = featurizer.featurize_dataframe(
    df,
    col_id="composition",
    ignore_errors=True
)

print("Feature generation completed!")

print("New dataset shape:",
      df_features.shape)

Generating Magpie features...


ElementProperty:   0%|          | 0/10000 [00:00<?, ?it/s]

Feature generation completed!
New dataset shape: (10000, 138)


In [8]:
# Get Magpie feature columns
magpie_columns = [
    col for col in df_features.columns
    if col not in df.columns
]

print("Number of generated features:",
      len(magpie_columns))

print("\nGenerated features:")
print(magpie_columns)

Number of generated features: 132

Generated features:
['MagpieData minimum Number', 'MagpieData maximum Number', 'MagpieData range Number', 'MagpieData mean Number', 'MagpieData avg_dev Number', 'MagpieData mode Number', 'MagpieData minimum MendeleevNumber', 'MagpieData maximum MendeleevNumber', 'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber', 'MagpieData avg_dev MendeleevNumber', 'MagpieData mode MendeleevNumber', 'MagpieData minimum AtomicWeight', 'MagpieData maximum AtomicWeight', 'MagpieData range AtomicWeight', 'MagpieData mean AtomicWeight', 'MagpieData avg_dev AtomicWeight', 'MagpieData mode AtomicWeight', 'MagpieData minimum MeltingT', 'MagpieData maximum MeltingT', 'MagpieData range MeltingT', 'MagpieData mean MeltingT', 'MagpieData avg_dev MeltingT', 'MagpieData mode MeltingT', 'MagpieData minimum Column', 'MagpieData maximum Column', 'MagpieData range Column', 'MagpieData mean Column', 'MagpieData avg_dev Column', 'MagpieData mode Column', 'MagpieData 

In [9]:
# ============================================================
# CREATE X AND y
# ============================================================

X = df_features[magpie_columns].copy()

y = df_features[target_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nX:")
display(X.head())

print("\ny:")
display(y.head())

X shape: (10000, 132)
y shape: (10000,)

X:


,MagpieData minimum Number,MagpieData maximum Number,MagpieData range Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData range MendeleevNumber,MagpieData mean MendeleevNumber,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,6.0,20.0,14.0,9.200000,2.720000,8.0,7.0,87.0,80.0,75.100000,...,0.0,0.0,0.0,0.0,12.0,225.0,213.0,87.900000,91.08000,12.0
1,6.0,20.0,14.0,9.466667,3.146667,8.0,7.0,87.0,80.0,73.066667,...,0.0,0.0,0.0,0.0,12.0,225.0,213.0,88.933333,92.32000,12.0
2,6.0,20.0,14.0,9.200000,2.720000,8.0,7.0,87.0,80.0,75.100000,...,0.0,0.0,0.0,0.0,12.0,225.0,213.0,87.900000,91.08000,12.0
3,3.0,17.0,14.0,12.823529,4.913495,17.0,1.0,94.0,93.0,67.529412,...,0.0,0.0,0.0,0.0,64.0,229.0,165.0,125.764706,72.66436,64.0
4,6.0,20.0,14.0,8.800000,2.080000,8.0,7.0,87.0,80.0,78.150000,...,0.0,0.0,0.0,0.0,12.0,225.0,213.0,86.350000,89.22000,12.0



y:


0    Monoclinic
1    Monoclinic
2     Triclinic
3    Monoclinic
4     Triclinic
Name: crystal_system, dtype: object

In [10]:
# ============================================================
# CLEAN NUMERICAL FEATURES
# ============================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

# Convert all columns to numeric
X = X.apply(
    pd.to_numeric,
    errors="coerce"
)

# Remove columns containing only NaN
X = X.dropna(
    axis=1,
    how="all"
)

# Fill missing values using median
X = X.fillna(
    X.median()
)

print("Final numerical X shape:", X.shape)

print("\nMissing values remaining:",
      X.isna().sum().sum())

Final numerical X shape: (10000, 132)

Missing values remaining: 0


In [11]:
from sklearn.feature_selection import VarianceThreshold

# ============================================================
# VARIANCE FILTER
# ============================================================

variance_filter = VarianceThreshold(
    threshold=0.01
)

X_variance_array = variance_filter.fit_transform(X)

selected_columns = X.columns[
    variance_filter.get_support()
]

X_variance = pd.DataFrame(
    X_variance_array,
    columns=selected_columns
)

print("Features before variance filtering:",
      X.shape[1])

print("Features after variance filtering:",
      X_variance.shape[1])

Features before variance filtering: 132
Features after variance filtering: 128


In [12]:
# ============================================================
# CORRELATION FILTER
# ============================================================

correlation_matrix = X_variance.corr().abs()

upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

correlation_threshold = 0.95

high_correlation_columns = [
    column
    for column in upper_triangle.columns
    if any(
        upper_triangle[column] > correlation_threshold
    )
]

X_filtered = X_variance.drop(
    columns=high_correlation_columns
)

print("Features before correlation filtering:",
      X_variance.shape[1])

print("Highly correlated features removed:",
      len(high_correlation_columns))

print("Features after correlation filtering:",
      X_filtered.shape[1])

Features before correlation filtering: 128
Highly correlated features removed: 26
Features after correlation filtering: 102


In [13]:
# ============================================================
# FINAL DATASET
# ============================================================

X = X_filtered.reset_index(drop=True)

y = y.reset_index(drop=True)

print("========================================")
print("FINAL PHASE 3 DATA")
print("========================================")

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget classes:")
print(y.value_counts())

print("\nFirst 5 X rows:")
display(X.head())

print("\nFirst 5 y values:")
display(y.head())

FINAL PHASE 3 DATA
X shape: (10000, 102)
y shape: (10000,)

Target classes:
crystal_system
Triclinic       4230
Monoclinic      2601
Trigonal        1027
Orthorhombic     911
Cubic            620
Tetragonal       540
Hexagonal         71
Name: count, dtype: int64

First 5 X rows:


,MagpieData minimum Number,MagpieData maximum Number,MagpieData mean Number,MagpieData avg_dev Number,MagpieData mode Number,MagpieData minimum MendeleevNumber,MagpieData maximum MendeleevNumber,MagpieData mean MendeleevNumber,MagpieData avg_dev MendeleevNumber,MagpieData mode MendeleevNumber,...,MagpieData avg_dev GSbandgap,MagpieData mode GSbandgap,MagpieData maximum GSmagmom,MagpieData mean GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,6.0,20.0,9.200000,2.720000,8.0,7.0,87.0,75.100000,15.040000,87.0,...,1.438720,0.000,0.0,0.0,12.0,225.0,213.0,87.900000,91.08000,12.0
1,6.0,20.0,9.466667,3.146667,8.0,7.0,87.0,73.066667,18.293333,87.0,...,1.438720,0.000,0.0,0.0,12.0,225.0,213.0,88.933333,92.32000,12.0
2,6.0,20.0,9.200000,2.720000,8.0,7.0,87.0,75.100000,15.040000,87.0,...,1.438720,0.000,0.0,0.0,12.0,225.0,213.0,87.900000,91.08000,12.0
3,3.0,17.0,12.823529,4.913495,17.0,1.0,94.0,67.529412,31.307958,94.0,...,1.207682,2.493,0.0,0.0,64.0,229.0,165.0,125.764706,72.66436,64.0
4,6.0,20.0,8.800000,2.080000,8.0,7.0,87.0,78.150000,10.620000,87.0,...,1.438720,0.000,0.0,0.0,12.0,225.0,213.0,86.350000,89.22000,12.0



First 5 y values:


0    Monoclinic
1    Monoclinic
2     Triclinic
3    Monoclinic
4     Triclinic
Name: crystal_system, dtype: object

In [14]:
# ============================================================
# SAVE FEATURES
# ============================================================

features_output = (
    r"C:\Users\Atul\materials_project_cleaning"
    r"\matminer_magpie_features.csv"
)

X.to_csv(
    features_output,
    index=False
)

print("Features saved successfully:")
print(features_output)

Features saved successfully:
C:\Users\Atul\materials_project_cleaning\matminer_magpie_features.csv
